# Aplicación del modelo final a test

Aplica lo que guardó `entrenar_modelo_final.ipynb` en `modelo_final/` y genera `submissions/ensamble_fable_v9.csv`,
el archivo enviado a la competencia.

Necesita las features de test (`cache/full_te.parquet`), generadas en el paso 1 del entrenamiento. Algunas son agregados sobre train+test o encodings ajustados con todo train, así que forman parte del pipeline entrenado.

In [1]:
import glob, json, joblib
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
import features as F
SEEDS = [42, 43, 44]
_, X_test = F.load_full()   # features de test, con las columnas de los transformers (gpu_kaggle/salidas)

## Ensamble: boostings guardados + TabPFN

In [2]:
def xgb_model(path):
    m = xgb.XGBClassifier(); m.load_model(path)
    return m


cats = [c for c in X_test.columns if str(X_test[c].dtype) == "category"]
X_test_cb = X_test.copy(); X_test_cb[cats] = X_test_cb[cats].astype(str)   # CatBoost recibe las categoricas como texto
boost = np.mean([
    np.mean([lgb.Booster(model_file=f"modelo_final/lightgbm_seed{s}.txt").predict(X_test) for s in SEEDS], axis=0),
    np.mean([xgb_model(f"modelo_final/xgboost_seed{s}.json").predict_proba(X_test)[:, 1] for s in SEEDS], axis=0),
    np.mean([CatBoostClassifier().load_model(f"modelo_final/catboost_seed{s}.cbm").predict_proba(X_test_cb)[:, 1] for s in SEEDS], axis=0),
], axis=0)
tabpfn_test = np.load("gpu_kaggle/salidas/tabpfn.npz")["test"]   # TabPFN con todo train como contexto (Kaggle)
ens = 0.5 * tabpfn_test + 0.5 * boost

## Combinación con el LLM por tramos

El historial del vendedor de cada publicación de test se cuenta en todo train.

In [3]:
lg = lambda q: np.log(np.clip(q, .01, .99) / (1 - np.clip(q, .01, .99)))
train, test = F.load("train_data.jsonlines"), F.load("test_data.jsonlines")
g = pd.Series((train[F.TARGET] == "new").astype(int).values).groupby(train["seller_id"].values).agg(["mean", "size"])
hist = test["seller_id"].map(g["size"]).fillna(0).values
pur = test["seller_id"].map(g["mean"]).values

TRAMOS = {  # nombre: (filtro, version del LLM, corrida de test)
    "nuevos": (lambda h, p: h == 0, "claude-opus-5-5", "test_nuevos"),
    "pocos": (lambda h, p: (h >= 1) & (h <= 3), "claude-opus-5-5", "test_pocos"),
    "mixtos": (lambda h, p: (h >= 4) & (p >= .2) & (p <= .8), "claude-fable-5-2-v3", "test_mixtos"),
}
combinacion = joblib.load("modelo_final/combinacion_llm.joblib")
proba = ens.copy()
for nombre, (filtro, version, parte) in TRAMOS.items():
    P = {}
    for f in glob.glob(f"llm/puntajes/{version}/{parte}/*.json"):
        P.update({int(k): v / 100 for k, v in json.load(open(f, encoding="utf8"))["p"].items()})
    rows = np.flatnonzero(filtro(hist, pur))
    llm = np.array([P[len(train) + r] for r in rows])   # en los puntajes, la fila r de test tiene id len(train) + r
    proba[rows] = combinacion[nombre].predict_proba(np.c_[lg(ens[rows]), lg(llm)])[:, 1]
    print(f"{nombre}: {len(rows)} publicaciones de test | cambian {((proba[rows] > .5) != (ens[rows] > .5)).sum()} predicciones")

nuevos: 8513 publicaciones de test | cambian 233 predicciones


pocos: 6043 publicaciones de test | cambian 63 predicciones
mixtos: 1936 publicaciones de test | cambian 39 predicciones


In [4]:
sub = F.save_submission(proba, "ensamble_fable_v9")
ref = pd.read_csv("submissions/referencia_ensamble_fable_v9.csv")
print("idéntico al archivo enviado:", sub.equals(ref), "| predicciones distintas:", (sub["condition"] != ref["condition"]).sum())
sub["condition"].value_counts(normalize=True)

idéntico al archivo enviado: True | predicciones distintas: 0


condition
new     0.527833
used    0.472167
Name: proportion, dtype: float64